In [34]:
import os
import joblib
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import xgboost as xgb
from lightgbm import LGBMRegressor
import lightgbm as lgb
from sklearn.base import ClassifierMixin, RegressorMixin
from sklearn.preprocessing import FunctionTransformer

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

def load_pipeline_model(model_path: str) -> Pipeline:
    """
    Charge une Pipeline sklearn depuis un fichier .joblib dont le nom
    encode le label modélisé. Vérifie que l'objet chargé est bien
    une sklearn.pipeline.Pipeline et signale la présence d'un pas XGBRegressor.
    
    Parameters
    ----------
    model_path : str
        Chemin vers le fichier .joblib (ex. '.../xgbregressor_normalized_ENB2012_data.csv_l_heating_load.joblib')
    
    Returns
    -------
    pipeline : sklearn.pipeline.Pipeline
        La pipeline chargée.
    
    Raises
    ------
    ValueError
        Si le nom de fichier n'est pas au format attendu.
    TypeError
        Si l'objet chargé n'est pas une Pipeline.
    """
    # 1. Extraction du label depuis le nom de fichier
    basename    = os.path.basename(model_path)
    if basename.endswith('.joblib'):
        
        
        name_no_ext = basename[:-7]  # enlève '.joblib'
        try:
            # on ignore la première partie (classe) avant le premier '_'
            _, label  = name_no_ext.split('_', 1)
        except ValueError:
            raise ValueError(
                f"Le nom '{basename}' n'est pas au format attendu "
                "(<classe>_<label>.joblib)"
            )
        
        # 2. Chargement
        pipeline = joblib.load(model_path)
        
        # 3. Vérification du type
        if not isinstance(pipeline, Pipeline):
            raise TypeError(
                f"Objet chargé de type {type(pipeline).__name__} inattendu, "
                "attendu sklearn.pipeline.Pipeline"
            )
        
        # 4. inspection des estimateurs
        found = []
        for name, step in pipeline.named_steps.items():
            if isinstance(step, (ClassifierMixin, RegressorMixin)):
                found.append((name, step.__class__.__name__))
    
        if found:
            print("✅ Estimateurs détectés dans la pipeline :")
            for step_name, cls_name in found:
                print(f"  - {step_name}: {cls_name}")
        else:
            print("⚠️ Aucune étape de modélisation scikit-learn trouvée.")
    else:
        raise ValueError(f"Le fichier '{basename}' doit se terminer par '.joblib'")
    # 5. On retourne simplement la Pipeline
    print(f"Pipeline chargée pour le label '{label}'.")
    return pipeline

In [35]:
# Création d'un DataFrame d'exemple
df_example = pd.DataFrame({
    'f_relative_compactness': [0.76, 0.85, 0.92, 0.65, 0.88,0.82],
    'f_wall_area': [270.0, 350.5, 210.2, 299.9, 330.0,318.5],
    'f_overall_height': [3.5, 7.0, 3.0, 5.5, 6.0,7.0],
    'f_orientation': [2, 3, 4, 5, 2, 5],
    'f_glazing_area': [0.0, 0.1, 0.2, 0.4, 0.3,0.0]
})
df_example

,f_relative_compactness,f_wall_area,f_overall_height,f_orientation,f_glazing_area
0,0.76,270.0,3.5,2,0.0
1,0.85,350.5,7.0,3,0.1
2,0.92,210.2,3.0,4,0.2
3,0.65,299.9,5.5,5,0.4
4,0.88,330.0,6.0,2,0.3
5,0.82,318.5,7.0,5,0.0


In [36]:
features_name = ['f_relative_compactness', 'f_wall_area', 'f_overall_height', 'f_glazing_area', 'f_orientation'
                # 'f_orientation_2', 'f_orientation_3', 'f_orientation_4', 'f_orientation_5'
                ]

def prepare_data(df: pd.DataFrame):
    """
    - Garde uniquement les colonnes dont le nom est contenu dans un feature de m_cooling_load
    - Renomme ces colonnes en les préfixant 'f_'
    - Convertit toutes les colonnes en float sauf 'f_orientation' en category
    - Extrait y_actual_heating_load et y_actual_cooling_load

    Returns:
        X: DataFrame prêt pour le modèle
        y_actual_heating_load: Series
        y_actual_cooling_load: Series
    """
    # 1) Récupère les noms de features LightGBM
    feature_names = features_name

    # 2) Filtre et préfixe
    orig_cols = [col for col in df.columns if any(col in feat for feat in feature_names)]
    X = df[orig_cols].copy()
    X.columns = ['f_' + col for col in orig_cols]

    # 3) Conversion des types
    for col in X.columns:
        if col == 'f_orientation':
            # on garde l'orientation en category
            X[col] = X[col].astype('category')
        else:
            # conversion numérique stricte
            X[col] = pd.to_numeric(X[col], errors='raise')

    # 4) Extraction des y
    y_actual_heating_load = pd.to_numeric(df['heating_load'], errors='raise')
    y_actual_cooling_load = pd.to_numeric(df['cooling_load'], errors='raise')

    return X, y_actual_heating_load, y_actual_cooling_load

In [37]:
# 1. Charger le fichier CSV avec séparateur ';'
df = pd.read_csv('../data/normalized_csv/normalized_ENB2012_data.csv', sep=';',decimal=',')

# 2) préparation X et y
X, y_h, y_c = prepare_data(df)
model_path_l_cooling_load = 'saved_models/lgbmregressor_l_cooling_load.joblib'
model_path_l_heating_load = 'saved_models/lgbmregressor_l_heating_load.joblib'




In [38]:
for path in (model_path_l_cooling_load, model_path_l_heating_load):
    pipe = load_pipeline_model(path)

    if hasattr(pipe, "feature_names_in_"):
        print(f"{os.path.basename(path)} → {list(pipe.feature_names_in_)}")
    else:
        print(f"{os.path.basename(path)} → attribut feature_names_in_ absent")

✅ Estimateurs détectés dans la pipeline :
  - model: LGBMRegressor
Pipeline chargée pour le label 'l_cooling_load'.
lgbmregressor_l_cooling_load.joblib → ['f_relative_compactness', 'f_wall_area', 'f_overall_height', 'f_orientation', 'f_glazing_area']
✅ Estimateurs détectés dans la pipeline :
  - model: LGBMRegressor
Pipeline chargée pour le label 'l_heating_load'.
lgbmregressor_l_heating_load.joblib → ['f_relative_compactness', 'f_wall_area', 'f_overall_height', 'f_orientation', 'f_glazing_area']


In [39]:
X

,f_relative_compactness,f_wall_area,f_overall_height,f_orientation,f_glazing_area
0,0.98,294.0,7.0,2,0.0
1,0.98,294.0,7.0,3,0.0
2,0.98,294.0,7.0,4,0.0
3,0.98,294.0,7.0,5,0.0
4,0.90,318.5,7.0,2,0.0
...,...,...,...,...,...
763,0.64,343.0,3.5,5,0.4
764,0.62,367.5,3.5,2,0.4
765,0.62,367.5,3.5,3,0.4
766,0.62,367.5,3.5,4,0.4


In [40]:
pipeline_cooling_load = load_pipeline_model(model_path_l_cooling_load)
pipeline_heating_load = load_pipeline_model(model_path_l_heating_load)

preds_log_cool = pipeline_cooling_load.predict(X)
preds_log_heat = pipeline_heating_load.predict(X)

✅ Estimateurs détectés dans la pipeline :
  - model: LGBMRegressor
Pipeline chargée pour le label 'l_cooling_load'.
✅ Estimateurs détectés dans la pipeline :
  - model: LGBMRegressor
Pipeline chargée pour le label 'l_heating_load'.


In [41]:
# 5) Inverse du log (ou log1p/boxcox selon le label)
preds_cool = np.exp(preds_log_cool)   # ou np.expm1 si log1p
preds_heat = np.exp(preds_log_heat)

# 6) RMSE
rmse_cool = np.sqrt(mean_squared_error(y_c, preds_cool))
rmse_heat = np.sqrt(mean_squared_error(y_h, preds_heat))

print(f"RMSE Cooling : {rmse_cool:.4f}")
print(f"RMSE Heating : {rmse_heat:.4f}")


RMSE Cooling : 1.5977
RMSE Heating : 0.4607


## 